**Проект 1**

Исследования 

Подключаемся ко всей базе сразу для работы с более обширной выборкой

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
import itertools
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text
from sqlalchemy.exc import OperationalError

from prime.config import get_engine

engine = get_engine()
ACCENT, GREY = "#d62728", "#9a9a9a"
PRICE_CHANGE = pd.Timestamp("2026-03-01")
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})


def q(sql: str, retries: int = 8) -> pd.DataFrame:
    """Запрос -> DataFrame. База иногда обрывает новое соединение, поэтому повторяем."""
    for attempt in range(retries):
        try:
            with engine.connect() as conn:
                return pd.read_sql(text(sql), conn)
        except OperationalError:
            if attempt == retries - 1:
                raise
            engine.dispose()
            time.sleep(3 * (attempt + 1))


Делаем выгрузку для проверки существующих статусов платежей и проверяем, что id платежа нигде не повторяется

In [ ]:
pay_status = q("""
    select status, count(*) as n, count(distinct payment_id) as n_unique, sum(amount) as amount
    from prime.payments group by status order by n desc""")

# платёж -> день x тариф. С subscriptions соединяем ТОЛЬКО ради тарифа; subscription_members не трогаем
daily = q("""
    select p.paid_at::date as d, s.tariff_id, count(*) as n_pay, sum(p.amount) as rev
    from prime.payments p
    join prime.subscriptions s using (subscription_id)
    where p.status = 'success' and p.paid_at >= '2025-01-01' and p.paid_at < '2026-08-01'
    group by 1, 2 order by 1, 2""")
daily["d"] = pd.to_datetime(daily["d"])

price_check = q("""
    select s.tariff_id, p.amount, count(*) as n, min(p.paid_at)::date as first_day, max(p.paid_at)::date as last_day
    from prime.payments p join prime.subscriptions s using (subscription_id)
    where p.status = 'success' group by 1, 2 order by 1, 2""")

churn = q("""
    with m as (select generate_series('2025-01-01'::date, '2026-07-01'::date, '1 month')::date as month_start)
    select m.month_start as month, s.tariff_id,
      count(*) filter (where s.started_at < m.month_start and (s.ended_at is null or s.ended_at >= m.month_start)) as active_start,
      count(*) filter (where s.started_at < m.month_start and s.ended_at >= m.month_start
                         and s.ended_at < m.month_start + interval '1 month') as churned,
      count(*) filter (where s.started_at >= m.month_start and s.started_at < m.month_start + interval '1 month') as new_subs
    from m cross join prime.subscriptions s
    where s.started_at < m.month_start + interval '1 month'
    group by 1, 2 order by 1, 2""")
churn["month"] = pd.to_datetime(churn["month"])

reasons = q("""
    select date_trunc('month', ended_at)::date as month, cancel_reason, count(*) as n
    from prime.subscriptions where ended_at >= '2025-09-01' and ended_at < '2026-08-01'
    group by 1, 2""")
reasons["month"] = pd.to_datetime(reasons["month"])
pay_status


Проверяем статусы и их распределение, а также стоимость платежей по подписке

In [ ]:
assert (pay_status["n"] == pay_status["n_unique"]).all(), "дубли payment_id"
print("доли статусов:\n", (pay_status.set_index("status")["n"] / pay_status["n"].sum()).round(3))


def avg_price(df):
    return df.groupby("tariff_id").apply(lambda x: x["rev"].sum() / x["n_pay"].sum(), include_groups=False)


print("средняя цена платежа до/после:\n",
      pd.DataFrame({"до": avg_price(daily[daily["d"] < PRICE_CHANGE]),
                    "после": avg_price(daily[daily["d"] >= PRICE_CHANGE])}))

# на каждый тариф ровно две цены: одна до границы, одна после
for t, grp in price_check.groupby("tariff_id"):
    old = grp[grp["last_day"] < PRICE_CHANGE.date()]
    new = grp[grp["first_day"] >= PRICE_CHANGE.date()]
    assert len(old) == 1 and len(new) == 1 and len(grp) == 2, f"тариф {t}: цен больше двух"
price_check


**Как мы считаем: зерно, период и сегменты**

**Зерно** (что означает одна строка таблицы) выбрано под задачу каждого шага:

| Таблица | Одна строка = | Зачем |
|---|---|---|
| `daily` | день × тариф | выручка и платежи, декомпозиция роста |
| `reasons` | месяц × причина отмены | доля причин отмены |
| `cube` | месяц × тариф × канал | ARPU, отток и pLTV по сегментам |

Единица анализа во всех метриках (ARPU, отток, pLTV) — **подписка**, а не клиент и не платёж: у клиента может быть несколько подписок. Поэтому pLTV — это ценность подписки, а не клиента.

Зерно *день × тариф* выбрано по двум причинам. Во-первых, в месяцах разное число дней (февраль — 28, март — 31), поэтому месяцы сравниваем по выручке *в день*. Во-вторых, платёж слишком мелкая единица: их миллионы, а для графиков и декомпозиции достаточно дневных итогов. Канал привлечения есть только в `cube`: он нужен для сегментов pLTV, а в декомпозиции роста фактором не является.

Платежи соединяем только с `subscriptions` (ради тарифа). Таблицу `subscription_members` не используем: у подписки может быть несколько участников, и при таком соединении платёж размножался бы, а выручка завышалась.

**Период.** Данные берём с января 2025 по июль 2026. Август 2026 не включаем: данные в базе обрываются 30.08, и последний месяц был бы неполным. Подорожание произошло 01.03.2026, поэтому:
- Для разложения роста выручки сравниваем февраль и март 2026 (соседние месяцы);
- Для оттока, ARPU и pLTV сравниваем два окна по три месяца: до подорожания (декабрь 2025 – февраль 2026) и после (март – май 2026). Показатели приводим к 30 дням, чтобы разная длина месяцев не искажала сравнение;
- В апреле 2026 нет данных за 14–15.04. Выручку апреля доводим до полного месяца, иначе ARPU «после» был бы занижен. На отток эта дыра не влияет: он считается по датам окончания подписок, а не по платежам. Отток везде считаем одинаково: число подписок, закончившихся за период, делённое на подписко-дни (сколько дней подписки были активны в этом периоде), и приведённое к 30 дням. На графике по месяцам это значение за один месяц, в pLTV — за окно из трёх месяцев целиком. Так в отток входят и подписки, начатые внутри месяца, и показатель не зависит от длины месяца.

**Сегменты.** Смотрим на тариф (Базовый и Расширенный) и канал привлечения. Тариф важен, потому что цена выросла на разную долю (+25% и +12,5%). Канал нужен, чтобы проверить, не скрывает ли общая цифра разные истории. Клиентов без канала не исключаем, а выделяем в группу «нет канала»: иначе при группировке они пропали бы из расчёта.

**Что проверено** (ячейка ниже): в `daily` нет дублей ключа день × тариф, а число платежей и сумма выручки совпадают с запросом по `payments` без соединений.


In [ ]:
# Контроль зерна: ключ день × тариф уникален; число платежей и выручка равны запросу по payments без соединений
assert not daily.duplicated(["d", "tariff_id"]).any(), "в daily дубли ключа день × тариф"

ctrl = q("""select count(*) as n, sum(amount) as rev from prime.payments
            where status = 'success' and paid_at >= '2025-01-01' and paid_at < '2026-08-01'""").iloc[0]
assert daily["n_pay"].sum() == ctrl["n"], "join с subscriptions потерял платежи"
assert abs(daily["rev"].sum() - ctrl["rev"]) < 0.01, "выручка в daily не равна выручке в payments"
print(f"платежей {ctrl['n']:,.0f}, выручка {ctrl['rev']:,.0f} ₽: daily равна payments".replace(",", " "))


Видим, что в марте 2026 года произошло подорожание на обоих тарифных планах:
- На базовом цена выросла с 199 р до 249 р
- На расширенном тарифе цена выросла с 399 р до 449 р

Есть задание от продактов на исследование итогового эффекта на бизнес метрики и на доходность подписочного стрима. 

Заказчик хочет понять: заработали ли мы что-то с этого в моменте и не потеряли ли в долгосроке? 

Для того, чтобы оценить выиграли ли в моменте надо посмотреть как изменилась выручка в месяц по обоим тарифам, как изменилась дневная выручка (в месяце разное количество дней и важно понимать среднюю для адекватного измерения эффекта) и какой прирост по части выручки случился.

In [ ]:
daily["month"] = daily["d"].dt.to_period("M").dt.to_timestamp()
days_in_data = daily.groupby("month")["d"].nunique()      # дни с платежами: общие для обоих тарифов

monthly_t = daily.groupby(["month", "tariff_id"]).agg(rev=("rev", "sum"), n_pay=("n_pay", "sum")).reset_index()
monthly_t["days"] = monthly_t["month"].map(days_in_data)
monthly_t["rev_per_day"] = monthly_t["rev"] / monthly_t["days"]
monthly_t["pay_per_day"] = monthly_t["n_pay"] / monthly_t["days"]

TARIFF_STYLE = {1: ("Базовый", GREY), 2: ("Расширенный", ACCENT)}
TARIFF_NAMES = {t: name for t, (name, _) in TARIFF_STYLE.items()}

rev_m = monthly_t.pivot(index="month", columns="tariff_id", values="rev") / 1e6
rev_d = monthly_t.pivot(index="month", columns="tariff_id", values="rev_per_day") / 1e6

levels = pd.concat({"выручка за месяц, млн ₽": rev_m.rename(columns=TARIFF_NAMES),
                    "выручка в день, млн ₽": rev_d.rename(columns=TARIFF_NAMES)}, axis=1)
mom = pd.concat({"MoM выручки за месяц, %": rev_m.pct_change().rename(columns=TARIFF_NAMES) * 100,
                 "MoM выручки в день, %": rev_d.pct_change().rename(columns=TARIFF_NAMES) * 100}, axis=1)
display(pd.concat([levels, mom], axis=1).round(2).tail(8))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for ax, data, title, ylabel in [(axes[0], rev_m, "Выручка за месяц", "млн ₽"),
                                (axes[1], rev_d, "Выручка в день", "млн ₽ в день")]:
    for t, (name, col) in TARIFF_STYLE.items():
        ax.plot(data.index, data[t], marker="o", ms=4, color=col, label=name)
    ax.axvline(PRICE_CHANGE, ls="--", color="black", lw=1)
    ax.set(title=title, ylabel=ylabel, ylim=(0, None))
    ax.tick_params(axis="x", rotation=45)
axes[0].legend(); plt.tight_layout(); plt.show()


def change(a, b, t):
    x, y = (monthly_t[(monthly_t.month == m) & (monthly_t.tariff_id == t)].iloc[0] for m in (a, b))
    return {"выручка": y.rev / x.rev - 1, "платежей": y.n_pay / x.n_pay - 1,
            "выручка/день": y.rev_per_day / x.rev_per_day - 1, "платежей/день": y.pay_per_day / x.pay_per_day - 1}


table = pd.DataFrame({name: change("2026-02-01", "2026-03-01", t) for t, (name, _) in TARIFF_STYLE.items()})
table.index.name = "Фев → Мар, %"
display((table * 100).round(1))


Что мы видим: в марте значительный рост выручки MtM не только месячном разрезе, но и что важнее в среднедневном разрезе по сравнению с февралём. Теперь проверим нет ли влияния на выручку из-за количества списаний. Нельзя исключать что причиной роста стало внезапное привлечение большого числа пользователей. 

In [ ]:
dd = daily.groupby("d")["n_pay"].sum().loc["2025-12-01":]
fig, ax = plt.subplots()
ax.plot(dd.index, dd.values, color=GREY, lw=1)
ax.axvline(PRICE_CHANGE, ls="--", color=ACCENT, lw=1)
ax.set(title="Платежей в день растёт плавно, на 01.03 излома нет",
       ylabel="Успешных платежей в день", ylim=(0, None))
plt.tight_layout(); plt.show()

wd = dd.groupby(dd.index.dayofweek).mean().round(0)
wd.index = ["пн", "вт", "ср", "чт", "пт", "сб", "вс"]
wd


График подтверждает, что количество платежей в день расчёт плавно и причиной кратного роста выручки являться не может. 
Итого: выяснили, что количество дней в месяце и количество платежей влияют, но не так значительно как подорожание тарифов. Декомпозируем и посмотрим с помощью каскадной (waterfall) диаграммы как по факторам разделилось влияние на рост цены. Для упрощения выделены 3 категории: длина месяца, рост среднего количества платежей в день и рост стоимости тарифов.



In [ ]:
# Декомпозиция роста выручки: февраль → март 2026
M0, M1 = "2026-02", "2026-03"


def month_totals(df, month):
    """Итоги месяца: дни, платежей, выручка, платежей в день, средний чек."""
    x = df[df["d"].dt.to_period("M") == month]
    days, n, rev = x["d"].nunique(), x["n_pay"].sum(), x["rev"].sum()
    assert days == pd.Period(month).days_in_month, f"{month}: в данных есть дни без платежей"
    return {"days": days, "n": n, "rev": rev, "per_day": n / days, "check": rev / n}


a, b = month_totals(daily, M0), month_totals(daily, M1)

# Цепные подстановки. Порядок (как в курсе: сначала количество, потом цена): дни → платежей в день → средний чек.
# Выручка = дни × платежей в день × средний чек.
eff = {
    "Длина месяца (дни)": (b["days"] - a["days"]) * a["per_day"] * a["check"],
    "Платежей в день": b["days"] * (b["per_day"] - a["per_day"]) * a["check"],
    "Средний чек (цена)": b["days"] * b["per_day"] * (b["check"] - a["check"]),
}
total = b["rev"] - a["rev"]
assert abs(sum(eff.values()) - total) < 0.01, "сумма влияний не равна изменению выручки"

# Микс тарифов в «среднем чеке»: сколько из его изменения дал сдвиг между тарифами, а не цена
t0 = daily[daily["d"].dt.to_period("M") == M0].groupby("tariff_id")[["n_pay", "rev"]].sum()
t1 = daily[daily["d"].dt.to_period("M") == M1].groupby("tariff_id")[["n_pay", "rev"]].sum()
s0, s1, p0, p1 = t0.n_pay / t0.n_pay.sum(), t1.n_pay / t1.n_pay.sum(), t0.rev / t0.n_pay, t1.rev / t1.n_pay
mix_part = ((s1 - s0) * p0).sum()                       # вклад смены долей тарифов в изменение среднего чека, ₽
price_part = (s1 * (p1 - p0)).sum()                     # вклад смены цен
assert abs(mix_part + price_part - (b["check"] - a["check"])) < 1e-6
print(f"выручка: {a['rev']/1e6:.2f} -> {b['rev']/1e6:.2f} млн ₽, изменение {total/1e6:+.2f}")
print(f"доля Базового тарифа в платежах: {s0[1]:.1%} -> {s1[1]:.1%}; вклад микса в изменение чека: {mix_part:+.2f} ₽ из {b['check'] - a['check']:+.2f} ₽")

# --- Водопад (ось от нуля, как требует Л6)
labels = ["Выручка\nфевраль", *eff.keys(), "Выручка\nмарт"]
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(0, a["rev"] / 1e6, color=GREY, zorder=2)
level = a["rev"] / 1e6
for i, (name, v) in enumerate(eff.items(), start=1):
    v_m = v / 1e6
    ax.bar(i, v_m, bottom=level, color=ACCENT if "чек" in name else "#b9bdc2", zorder=2)
    ax.text(i, level + v_m + 0.8, f"{v_m:+.2f} ({v / total:.0%})".replace(".", ","), ha="center", fontsize=10, fontweight="bold" if "чек" in name else "normal")
    level += v_m
ax.bar(len(eff) + 1, b["rev"] / 1e6, color=GREY, zorder=2)
for x_, v in [(0, a["rev"]), (len(eff) + 1, b["rev"])]:
    ax.text(x_, v / 1e6 + 0.8, f"{v / 1e6:.1f}".replace(".", ","), ha="center", fontsize=10)
ax.set_xticks(range(len(labels)), labels)
ax.set(ylim=(0, b["rev"] / 1e6 * 1.12), ylabel="млн ₽")                          # ось от нуля
price_share, days_share = eff["Средний чек (цена)"] / total, eff["Длина месяца (дни)"] / total
ax.set_title(f"Выручка в марте выросла на {total / 1e6:.1f} млн ₽: {price_share:.0%} дала цена, {days_share:.0%} — длина месяца".replace(".", ","), loc="left", fontsize=12.5)
fig.text(0.01, -0.03, "Цепные подстановки, порядок: дни → платежей в день → средний чек. Февраль и март 2026, сутки UTC. "
         "Сумма влияний равна изменению выручки (проверено в коде).", fontsize=8.5, color="#6b7280")
plt.tight_layout(); plt.show()


Увидели, что дни месяца повлияли, но не так сильно как прирост из-за роста среднего чека. Здесь завершили.

Продукт выстраивается с одной простой целью: заработать как можно больше и как можно дольше. С больше разобрались, а что там с дольше? 

Проанализируем отток клиентов по итогам изменения цен тарифов. 

In [ ]:
churn["days"] = churn["month"].dt.days_in_month
churn["risk30"] = churn["churned"] / (churn["active_start"] * churn["days"]) * 30 * 100
rp = churn.pivot(index="month", columns="tariff_id", values="risk30")

fig, ax = plt.subplots()
for t, (name, color) in TARIFF_STYLE.items():
    ax.plot(rp.index, rp[t], marker="o", color=color, label=name)
ax.axvline(PRICE_CHANGE, ls="--", color="black", lw=1)
ax.set(title="Риск отмены почти не меняется после 01.03",
       ylabel="% подписок в месяц (на 30 дней)", ylim=(3.5, 4.5))
ax.legend(); plt.tight_layout(); plt.show()


Отток клиентов после марта и в последующие месяцы не увеличился (за исключением расширенного тарифа в апреле). Чтобы точно удостовериться в том, что оттекающие оттекали не по причине подорожания посмотрим на результаты опроса клиентов при отмене подписки.

In [ ]:
rs = reasons.pivot(index="month", columns="cancel_reason", values="n")
share = rs.div(rs.sum(axis=1), axis=0) * 100

# доля «слишком дорого» среди всех отмен: 5 месяцев до (окт–фев) и 5 после (мар–июл)
pre_rs, post_rs = rs.loc["2025-10-01":"2026-02-01"], rs.loc["2026-03-01":"2026-07-01"]
p0 = pre_rs["too_expensive"].sum() / pre_rs.to_numpy().sum()
p1 = post_rs["too_expensive"].sum() / post_rs.to_numpy().sum()

NAMES = {"too_expensive": "слишком дорого", "not_using": "не пользуются", "switched": "перешли к другому",
         "payment_failed": "не прошла оплата", "other": "другое"}
ORDER = ["too_expensive", "not_using", "switched", "payment_failed", "other"]      # «слишком дорого» внизу: общая базовая линия
COLORS = {"too_expensive": ACCENT, "not_using": "#9aa0a6", "switched": "#b9bdc2", "payment_failed": "#d3d6d9", "other": "#e6e8ea"}

x = np.arange(len(share))
fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(share))
for reason in ORDER:
    vals = share[reason].to_numpy()
    ax.bar(x, vals, bottom=bottom, width=0.78, color=COLORS[reason], zorder=2)
    for xi, b, v in zip(x, bottom, vals):
        ax.text(xi, b + v / 2, f"{v:.0f}", ha="center", va="center", fontsize=9,
                color="white" if reason == "too_expensive" else "#374151", fontweight="bold" if reason == "too_expensive" else "normal")
    ax.text(x[-1] + 0.52, bottom[-1] + vals[-1] / 2, NAMES[reason], va="center", fontsize=9,
            color=ACCENT if reason == "too_expensive" else "#6b7280", fontweight="bold" if reason == "too_expensive" else "normal")
    bottom += vals

split = list(share.index).index(PRICE_CHANGE) - 0.5
ax.axvline(split, ls="--", color="black", lw=1, zorder=3)
ax.text(split + 0.08, 101.5, "подорожание 01.03", fontsize=8, va="bottom")
ax.set_xticks(x, [d.strftime("%m.%y") for d in share.index])
ax.set_xlim(-0.6, x[-1] + 2.2)
ax.set_title(f"Доля отмен «слишком дорого» после подорожания не изменилась: {p0:.1%} → {p1:.1%}".replace(".", ","), pad=22)
ax.set(ylabel="% отмен месяца", ylim=(0, 100))
ax.spines["left"].set_visible(True)
fig.text(0.01, -0.02, "Отмены подписок по месяцу окончания, сутки UTC. Доля до: окт 2025 – фев 2026, после: мар – июл 2026.",
         fontsize=8, color="#6b7280")
plt.tight_layout(); plt.show()


График явно показывает, что причина оставшегося оттока не в подорожании. Теперь ключевое: помимо подтверждения роста выручки и подтверждения отсутствия влияния оттока посмотрим как изменились ключевые метрики доходности продукта: ARPU и pLTV.

Почему это нужно: доходность в моменте (ARPU) может быть выше, но необходимо посмотреть как изменение повлияет на ценность клиента для компании в течение жизни в продукте. 

Как считаем pLTV = ARPU / churn. ARPU — выручка на одну подписку за 30 дней, отток — доля подписок, закончившихся за 30 дней (определение то же, что в графике выше). Оба показателя считаем за окно из 3 месяцев целиком: до подорожания (дек 2025 – фев 2026) и после (мар – май 2026). 1 ÷ отток даёт средний срок жизни подписки, около 26 месяцев.

In [ ]:
# Данные для pLTV: выручка, подписко-дни и отмены по месяцам × тариф × канал в двух окнах сравнения
PRE = ("2025-12-01", "2026-03-01")       # до: дек 2025 – фев 2026 (прежние цены)
POST = ("2026-03-01", "2026-06-01")      # после: мар – май 2026 (новые цены)

# Выручка: платёж -> подписка (тариф) -> клиент (канал); все соединения «многие к одному», строк столько же, сколько платежей.
rev_m = q("""
    select date_trunc('month', p.paid_at)::date as month, s.tariff_id,
           coalesce(c.acquisition_channel, 'нет канала') as channel, sum(p.amount) as rev
    from prime.payments p
    join prime.subscriptions s using (subscription_id)
    join prime.clients c on c.client_id = s.owner_client_id
    where p.status = 'success' and p.paid_at >= '2025-12-01' and p.paid_at < '2026-06-01'
    group by 1, 2, 3""")

# Экспозиция: сколько дней каждая подписка была активна в месяце (подписко-дни) и сколько подписок закончилось в месяце.
expo_m = q("""
    with m as (select generate_series('2025-12-01'::date, '2026-05-01'::date, '1 month')::date as ms)
    select m.ms as month, s.tariff_id, coalesce(c.acquisition_channel, 'нет канала') as channel,
           sum(extract(epoch from (least(coalesce(s.ended_at, m.ms + interval '1 month'), m.ms + interval '1 month')
                                   - greatest(s.started_at, m.ms))) / 86400.0) as sub_days,
           count(*) filter (where s.ended_at >= m.ms and s.ended_at < m.ms + interval '1 month') as churned
    from m cross join prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    where s.started_at < m.ms + interval '1 month' and (s.ended_at is null or s.ended_at >= m.ms)
    group by 1, 2, 3""")
for df in (rev_m, expo_m):
    df["month"] = pd.to_datetime(df["month"])

cube = expo_m.merge(rev_m, on=["month", "tariff_id", "channel"], how="left", validate="1:1").fillna({"rev": 0})
control = q("""select sum(amount) as rev from prime.payments
               where status = 'success' and paid_at >= '2025-12-01' and paid_at < '2026-06-01'""").iloc[0, 0]
assert abs(cube["rev"].sum() - control) < 0.01, "выручка в кубе не равна контролю по payments (потеряны или размножены строки)"

# В апреле 2026 нет данных за 14–15.04: доводим выручку апреля до полного месяца, иначе ARPU «после» занижен
obs_days = daily.groupby(daily["d"].dt.to_period("M").dt.to_timestamp())["d"].nunique()
fill = pd.Series(obs_days.index.days_in_month / obs_days.to_numpy(), index=obs_days.index)       # во сколько раз месяц длиннее дней с данными
fill[fill.index < pd.Timestamp(PRE[0])] = 1.0                                                    # только окна сравнения
cube["rev_full"] = cube["rev"] * cube["month"].map(fill)


def window_stats(by, win):
    """ARPU и отток за окно целиком: вся выручка (отмены) окна ÷ все подписко-дни окна × 30."""
    m = cube["month"]
    g = cube[(m >= win[0]) & (m < win[1])].groupby(by).agg(rev=("rev_full", "sum"), churned=("churned", "sum"), sub_days=("sub_days", "sum"))
    g["churn"] = g.churned / g.sub_days * 30              # отмен на подписку за 30 дней
    g["arpu"] = g.rev / g.sub_days * 30                   # выручка на подписку за 30 дней
    return g


def ltv_table(by):
    """pLTV = ARPU ÷ отток (формула из Л5): сколько выручки подписка приносит за весь срок жизни 1 ÷ отток."""
    a, b = window_stats(by, PRE), window_stats(by, POST)
    g = pd.DataFrame({"arpu_old": a.arpu, "churn_old": a.churn, "arpu": b.arpu, "churn": b.churn})
    g["pltv_old"] = g.arpu_old / g.churn_old
    g["pltv"] = g.arpu / g.churn
    g["lifetime_m"] = 1 / g.churn                         # средний срок жизни подписки, месяцев
    return g


ltv_ct, ltv_ch, ltv_t = ltv_table(["channel", "tariff_id"]), ltv_table("channel"), ltv_table("tariff_id")


In [ ]:
# --- График 1: ARPU и pLTV, до и после подорожания (в т.ч. общий по всем подпискам)
cube["all"] = "Все"
ltv_all = ltv_table("all")                                     # общий итог: все тарифы и каналы вместе
ALL = "Все каналы"
order = ltv_ch["pltv"].sort_values(ascending=False).index     # каналы от самого ценного
rows = list(order) + [ALL]

fig = plt.figure(figsize=(18, 5.9))
gs = fig.add_gridspec(1, 2, width_ratios=[1.5, 4.6], wspace=0.3)                 # слева ARPU, справа три панели pLTV
ax_arpu = fig.add_subplot(gs[0])
gs_ltv = gs[1].subgridspec(1, 3, wspace=0.08)
ax_b = fig.add_subplot(gs_ltv[0])
ax_e = fig.add_subplot(gs_ltv[1], sharex=ax_b, sharey=ax_b)
ax_all = fig.add_subplot(gs_ltv[2], sharex=ax_b, sharey=ax_b)

LAB_PRE, LAB_POST = "до: дек 2025 – фев 2026 (прежние цены)", "после: мар – май 2026 (новые цены)"
groups = [(TARIFF_NAMES[1], ltv_t.loc[1]), (TARIFF_NAMES[2], ltv_t.loc[2]), ("Все", ltv_all.loc["Все"])]
x = np.arange(len(groups)); w = 0.36
for off, col, color, lab in [(-w / 2, "arpu_old", GREY, LAB_PRE), (w / 2, "arpu", ACCENT, LAB_POST)]:
    vals = [g[col] for _, g in groups]
    ax_arpu.bar(x + off, vals, width=w, color=color, label=lab, zorder=2)
    for xi, v in zip(x + off, vals):
        ax_arpu.text(xi, v + 8, f"{v:.0f}", ha="center", fontsize=9)
for xi, (_, g) in zip(x, groups):
    d = g["arpu"] / g["arpu_old"] - 1
    ax_arpu.text(xi, g["arpu"] + 45, f"+{d:.1%}".replace(".", ","), ha="center", fontsize=10, color=ACCENT, fontweight="bold")
ax_arpu.set_xticks(x, [n for n, _ in groups], fontsize=9.5)
ax_arpu.set(ylim=(0, max(g["arpu"] for _, g in groups) * 1.22), ylabel="₽ на подписку за 30 дней")
ax_arpu.set_title("ARPU", loc="left", fontsize=11)


def panel_value(panel, ch, col):
    """Значение pLTV для строки панели: канал или итог «Все каналы»; панель: тариф 1, тариф 2 или все тарифы."""
    if panel == "all":
        return (ltv_all.loc["Все", col] if ch == ALL else ltv_ch.loc[ch, col])
    return ltv_t.loc[panel, col] if ch == ALL else ltv_ct.loc[(ch, panel), col]


xmax = max(ltv_ct["pltv"].max(), ltv_ch["pltv"].max()) * 1.2
for ax, panel, title in [(ax_b, 1, f"pLTV: {TARIFF_NAMES[1]}"), (ax_e, 2, f"pLTV: {TARIFF_NAMES[2]}"), (ax_all, "all", "pLTV: все тарифы")]:
    y = np.arange(len(rows))
    for off, col, color in [(-0.2, "pltv_old", GREY), (0.2, "pltv", ACCENT)]:
        vals = [panel_value(panel, ch, col) for ch in rows]
        ax.barh(y + off, vals, height=0.38, color=color, zorder=2)
        for yi, v in zip(y + off, vals):
            ax.text(v + xmax * 0.01, yi, f"{v:,.0f}".replace(",", " "), va="center", fontsize=8.5, color="#374151")
    ax.axhline(len(order) - 0.5, color="#d1d5db", lw=1)                    # линия над итоговой строкой
    ax.set_yticks(y, rows)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set(xlim=(0, xmax), xlabel="₽ выручки за срок жизни")
ax_b.invert_yaxis()                                              # оси общие: инвертируем один раз
ax_b.get_yticklabels()[-1].set_fontweight("bold")
plt.setp(ax_e.get_yticklabels(), visible=False)
plt.setp(ax_all.get_yticklabels(), visible=False)

arpu_all = ltv_all.loc["Все", "arpu"] / ltv_all.loc["Все", "arpu_old"] - 1
ltv_all_up = ltv_all.loc["Все", "pltv"] / ltv_all.loc["Все", "pltv_old"] - 1
fig.suptitle(f"После подорожания pLTV подписки вырос в среднем на {ltv_all_up:.0%} (ARPU на {arpu_all:.0%}): "
             f"Базовый на {ltv_t.loc[1, 'pltv'] / ltv_t.loc[1, 'pltv_old'] - 1:.0%}, Расширенный на {ltv_t.loc[2, 'pltv'] / ltv_t.loc[2, 'pltv_old'] - 1:.0%}".replace(".", ","),
             fontsize=12.5, x=0.01, ha="left")
handles, labels = ax_arpu.get_legend_handles_labels()            # цвета одинаковы во всех панелях
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2, frameon=False, fontsize=10)
fig.text(0.01, -0.12, "pLTV = ARPU ÷ отток. ARPU — выручка на подписку за 30 дней, отток — доля подписок, закончившихся за 30 дней; "
         "оба показателя посчитаны за 3 месяца окна целиком. 1 ÷ отток — средний срок жизни подписки (около 26 месяцев).\nПрогноз при условии, что ARPU и отток периода сохранятся; выручка без вычета кэшбэка. "
         "Сравнение реальных периодов, сутки UTC; апрель 2026 доведён до полного месяца (в данных нет 14–15.04).", fontsize=8.5, color="#6b7280", va="top")
plt.show()


Рост ARPU был ожидаемым: цена за тариф-то повысилась: на базовом рост 24,8%, на расширенном 12,3%.
Что важнее pLTV после подорожания вырос во всех каналах и во всех тарифах. 

Итого: рост стоимости подписки помог увеличить pLTV с 7080 рублей до 8400 рублей.



**Резюме исследования**

В рамках исследования проанализировано влияние роста стоимости подписки в марте 2026 на 2 тарифах: базовом и расширенном. 

Декомпозиция факторов показала, что рост стоимости тарифов обеспечил 61% M/M прироста выручки от подписок с февраля по март. 

Важно сказать, что мы можем не знать наверняка, что при прочих равных с 95% вероятностью такой же прирост был всегода, так как у нас не был проведен A/B-тест в котором сравнивались бы контроль с тестом и тест показал бы статзначимый результат. 

Далее мы проанализировали отток клиентов и выяснили, что отток нотток практически не изменился: около 3,9% подписок за 30 дней до подорожания и 3,8% после, то есть в пределах обычных колебаний. При этом цена подписки как фактор ухода клиента не выросла как в процентном соотоношении среди прочих причин отказа от подписки и сохранилась на уровне примерно равном 24,1%. 

Для проверки отсутствия негативного влияния повышения цен также посчитали pLTV и выяснили, что он не менялся в сторону снижения ни в одном из тарифов, а также не упал ни в одном из каналов привлечения трафика. 

По поводу рекомендаций: в рамках достижения цели повышения доходности направления стоит рассмотреть аккуратное повышение цен на подписки, если того позволяют среднерыночные цены и соотношение value для клиента / стоимость подписки. 